In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.functions import input_file_name
from pyspark.sql.functions import col, count, current_timestamp
import time
from delta.tables import DeltaTable

# ─────────────────────────────────────────────────────────────
# 1. Define Schema for Monitoring Delta Table
# ─────────────────────────────────────────────────────────────
monitor_schema = StructType([
    StructField("batch_id",           LongType(),      False),
    StructField("file_count",         IntegerType(),   True),
    StructField("record_count",       LongType(),      True),
    StructField("file_paths",         StringType(),    True),   # comma-separated file list
    StructField("batch_start_time",   TimestampType(), True),
    StructField("batch_end_time",     TimestampType(), True),
    StructField("duration_seconds",   LongType(),      True),
    StructField("status",             StringType(),    True),   # SUCCESS / FAILED
    StructField("error_message",      StringType(),    True),
])

order_schema=StructType([
    StructField("order_id", IntegerType(), True),
    StructField("order_date", TimestampType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("order_status", StringType(), True)
])

# ─────────────────────────────────────────────────────────────
# 2. Create Monitoring Delta Table (if not exists)
# ─────────────────────────────────────────────────────────────
monitor_table_path = "/Volumes/arn_unityc_dw/arn_dwbi/volume_archive_files"
monitor_table_name = "arn_unityc_dw.arn_dwbi.autoloader_batch_monitor"

# ─────────────────────────────────────────────────────────────
# 3. foreachBatch Function with Monitoring
# ─────────────────────────────────────────────────────────────
TARGET_TABLE      = "arn_unityc_dw.arn_dwbi.orders_raw"
TARGET_TABLE_PATH = "/Volumes/arn_unityc_dw/arn_dwbi/volume_output/orders_raw"

def process_and_monitor(df, batch_id):

    batch_start = time.time()
    start_ts    = current_timestamp()
    status        = "SUCCESS"
    error_message = None
    file_count    = 0
    record_count  = 0
    file_paths    = ""

    try:
        record_count = df.count()
        if "_metadata" in df.columns:
            files_df = df.selectExpr("_metadata.file_path as file_path").distinct()
            file_count = files_df.count()
            file_paths = ", ".join([row["file_path"] for row in files_df.collect()])
                
        print(f"""
        ┌──────────────────────────────────────────┐
        │  Batch ID      : {batch_id:<23} │
        │  Files         : {file_count:<23} │
        │  Records       : {record_count:<23} │
        └──────────────────────────────────────────┘
        """)

        # ── Write actual data to target Delta table ──
        (df.drop("_metadata")          # drop metadata before writing
           .write
           .format("delta")
           .mode("append")
           .option("mergeSchema", "true")
            .save("/Volumes/arn_unityc_dw/arn_dwbi/volume_output_bronze")
        )
    except Exception as e:
        status        = "FAILED"
        error_message = str(e)
        print(f" Batch {batch_id} failed: {e}")
        raise e

    finally:
        # ── Capture end time & duration ──
        batch_end        = time.time()
        duration_seconds = int(batch_end - batch_start)

        # ── Build monitoring record ──
        monitor_data = [(
            batch_id,
            file_count,
            record_count,
            file_paths,
            None,             # batch_start_time → use current_timestamp() via SQL
            None,             # batch_end_time
            duration_seconds,
            status,
            error_message
        )]

        monitor_df = spark.createDataFrame(monitor_data, schema=monitor_schema)

        # ── Add timestamps via withColumn ──
        monitor_df = (
            monitor_df
            .withColumn("batch_start_time", lit(int(batch_start)).cast("timestamp"))
            .withColumn("batch_end_time",   lit(int(batch_end)).cast("timestamp"))
        )

        # ── Write monitoring record to Delta ──
        (monitor_df.write
                   .format("delta")
                   .mode("append")
                   .save("/Volumes/arn_unityc_dw/arn_dwbi/volume_archive_files")
        )

df = (
  spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header","false")
    .schema(order_schema)
    .option("cloudFiles.useManagedFileEvents", "true")
    .option("cloudFiles.cleanSource", "delete")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("cloudFiles.maxFilesPerTrigger", 5) 
    .option("cloudFiles.schemaLocation","/Volumes/arn_unityc_dw/arn_dwbi/volume_meta_schema")
    .load("/Volumes/arn_unityc_dw/arn_dwbi/volume_input_raw")
)
# ----------------------------------
# Write data to Delta (bronze layer)
# ----------------------------------
query=(
    df.writeStream
      .foreachBatch(process_and_monitor)
      .option("checkpointLocation", "/Volumes/arn_unityc_dw/arn_dwbi/volume_meta_checkpoint").option("mergeSchema", "true")
      .trigger(availableNow=True)
      .start()
     
)
query.awaitTermination()